# M4 Pattern Library Demo
扫描真实 SPY 数据,展示 6 个 pattern 的信号分布。

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from spx_scanner.data_layer.loader import load_data
from spx_scanner.data_layer.resampler import resample_1m_to_3m
from spx_scanner.features import compute_all_features
from spx_scanner.patterns.registry import get_all_patterns

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── 加载数据 ──────────────────────────────────────────────────────────────
df1m = load_data('../data/spy_1min.parquet')
df3m = resample_1m_to_3m(df1m)
df = compute_all_features(df3m)

print(f'3min bars: {len(df)}  ({df.index[0].date()} → {df.index[-1].date()})')
print(f'Columns: {len(df.columns)}')

3min bars: 1820  (2026-04-06 → 2026-04-23)
Columns: 60


In [3]:
# ── 运行所有 patterns ──────────────────────────────────────────────────────
patterns = get_all_patterns('SPY')
print(f'Loaded {len(patterns)} patterns:', [p.name for p in patterns])

all_signals = []
for pat in patterns:
    try:
        sigs = pat.detect(df)
        all_signals.extend(sigs)
        print(f'  {pat.name}: {len(sigs)} signals')
    except Exception as e:
        print(f'  {pat.name}: ERROR - {e}')

print(f'\nTotal signals: {len(all_signals)}')

Loaded 6 patterns: ['failed_breakout', 'orb_breakout', 'squeeze_release', 'vwap_rejection', 'liquidity_sweep', 'last_hour_drift']
  failed_breakout: 20 signals
  orb_breakout: 131 signals
  squeeze_release: 5 signals
  vwap_rejection: 6 signals
  liquidity_sweep: 16 signals
  last_hour_drift: 112 signals

Total signals: 290


In [4]:
# ── 信号汇总 DataFrame ────────────────────────────────────────────────────
if all_signals:
    rows = [{
        'timestamp':  s.timestamp,
        'pattern':    s.pattern,
        'direction':  s.direction,
        'confidence': round(s.confidence, 3),
        'entry':      round(s.entry_price, 3),
        'stop':       round(s.stop_level, 3),
        'target':     round(s.target_level, 3) if s.target_level else None,
        'hold_min':   s.suggested_hold_min,
    } for s in all_signals]
    sig_df = pd.DataFrame(rows).sort_values('timestamp').reset_index(drop=True)
    print(sig_df.groupby(['pattern','direction']).size().to_string())
    sig_df.head(10)

pattern          direction
failed_breakout  call          1
                 put          19
last_hour_drift  call         81
                 put          31
liquidity_sweep  call          5
                 put          11
orb_breakout     call         75
                 put          56
squeeze_release  call          5
vwap_rejection   call          1
                 put           5


In [5]:
# ── 信号分布图 ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('M4 Pattern Signals — SPY 3-min (14 days)', fontsize=14, fontweight='bold')

# 1. 每个 pattern 信号数
counts = sig_df.groupby(['pattern','direction']).size().unstack(fill_value=0)
counts.plot(kind='bar', ax=axes[0], color=['#e74c3c','#27ae60'], edgecolor='white')
axes[0].set_title('Signal Count by Pattern & Direction')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Direction')

# 2. 置信度分布
for pat_name, grp in sig_df.groupby('pattern'):
    axes[1].hist(grp['confidence'], bins=10, alpha=0.5, label=pat_name, edgecolor='white')
axes[1].set_title('Confidence Distribution')
axes[1].set_xlabel('Confidence')
axes[1].legend(fontsize=7)

# 3. 按小时分布
sig_df['hour'] = pd.to_datetime(sig_df['timestamp']).dt.hour
hour_counts = sig_df.groupby(['hour','pattern']).size().unstack(fill_value=0)
hour_counts.plot(kind='bar', ax=axes[2], stacked=True, edgecolor='white')
axes[2].set_title('Signal Count by Hour')
axes[2].set_xlabel('Hour (ET)')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig('M4_signal_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved M4_signal_distribution.png')

Saved M4_signal_distribution.png


In [6]:
# ── 每个 pattern 展示 explain() 样例 ──────────────────────────────────────
pat_map = {p.name: p for p in patterns}
for pat_name, grp in sig_df.groupby('pattern'):
    sample_idx = grp.index[0]
    sig_obj = all_signals[[s.pattern for s in all_signals].index(pat_name)]
    print('=' * 60)
    print(pat_map[pat_name].explain(sig_obj))

[Failed Breakdown → CALL]
  支撑位: 654.85  触碰次数: 9
  RVOL: 1.84  BB%分位: 0.84
  EMA 粘合: True  连续上涨: 5 bar
  入场: 657.190  止损: 653.540  目标: 664.489  置信度: 0.80
[Last Hour Drift → CALL]
  距收盘: 78min  EMA21斜率: 0.2549
  RVOL: 1.0  在VWAP上方: True
  入场: 657.900  止损: 657.077  目标: 659.545  置信度: 0.60
[Liquidity Sweep → PUT]
  扫除关键位: prior_hour_high @ 653.885
  影线比: 0.689  RVOL: 1.6
  入场: 653.490  止损: 655.193  目标: 650.084  置信度: 0.50
[ORB Breakout → PUT]
  ORB High: 657.58  Low: 654.01
  RVOL: 1.55  ORB大小分位: 1.0
  入场: 652.600  止损: 657.580  目标: 645.130  置信度: 0.60
[Squeeze Release → CALL]
  压缩持续: 5 bar  BB%分位: 0.092
  RVOL: 1.69  ATR比: 0.495
  入场: 675.870  止损: 675.495  目标: 676.619  置信度: 0.65
[VWAP Rejection → PUT]
  VWAP: 655.825  影线比: 0.503
  RVOL: 1.34
  入场: 654.850  止损: 656.809  目标: 650.933  置信度: 0.60


In [7]:
# ── 用 viz.chart 展示一个代表性信号 ─────────────────────────────────────────
from spx_scanner.viz.chart import plot_signal_context

saved = []
shown_patterns = set()

for sig in all_signals:
    if sig.pattern in shown_patterns:
        continue
    # 过滤置信度 >= 0.6
    if sig.confidence < 0.6:
        continue
    try:
        fig = plot_signal_context(df, sig, bars_before=20, bars_after=15)
        fname = f'M4_review_{sig.pattern}_{sig.direction}.png'
        fig.savefig(fname, dpi=120, bbox_inches='tight')
        plt.close(fig)
        shown_patterns.add(sig.pattern)
        saved.append(fname)
        print(f'Saved {fname}')
    except Exception as e:
        print(f'  {sig.pattern}: chart error - {e}')
        shown_patterns.add(sig.pattern)

print(f'\nGenerated {len(saved)} review charts.')

Saved M4_review_failed_breakout_call.png
Saved M4_review_orb_breakout_put.png
Saved M4_review_squeeze_release_call.png
Saved M4_review_vwap_rejection_put.png
Saved M4_review_liquidity_sweep_put.png
Saved M4_review_last_hour_drift_call.png

Generated 6 review charts.


## M4 Summary

| Pattern | Priority | Description |
|---------|----------|-------------|
| `orb_breakout` | P0 | 开盘30min区间突破 |
| `failed_breakout` | P0 | 阻力/支撑位失败突破反转 |
| `squeeze_release` | P1 | BB压缩后单边爆发 |
| `vwap_rejection` | P1 | VWAP 触碰被拒绝 |
| `liquidity_sweep` | P1 | 扫前高/前低后反转 |
| `last_hour_drift` | P2 | 尾盘趋势顺势漂移 |

All 6 patterns registered, 145 tests passing. → M5 next.